# 02 — Auditoria de dados e reprodução dos resultados científicos

In [3]:
# Preparação: se executado a partir de um notebook aberto no Colab,
# carregue o ZIP do repositório gerado neste projeto (ou clone-o via GitHub).
from pathlib import Path
import os, sys, zipfile, subprocess
BASE = Path.cwd()
if not (BASE / 'iot_monitor').exists():
    git_repo = "https://github.com/jorgek14/railpulse-metropt3.git"
    if git_repo.strip():
        subprocess.run(['git', 'clone', git_repo, '/content/railpulse-metropt3'], check=True)
        BASE = Path('/content/railpulse-metropt3')
    else:
        try:
            from google.colab import files
        except ImportError as e:
            raise RuntimeError('Abra o notebook na raiz do repositório ou no Colab e envie o ZIP do projeto.') from e
        print('Selecione o ZIP do repositório RailPulse. NÃO selecione o CSV original agora.')
        enviados = files.upload()
        nome = next((n for n in enviados if n.endswith('.zip')), None)
        assert nome, 'É necessário carregar o ZIP do projeto.'
        BASE = Path('/content/railpulse-metropt3')
        BASE.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(nome) as z:
            z.extractall(BASE)
os.chdir(BASE)
sys.path.insert(0, str(BASE))
print('Raiz do projeto:', BASE)

Raiz do projeto: /content/railpulse-metropt3


In [6]:

from google.colab import drive
from pathlib import Path
import zipfile
import shutil

# 1. Conectar ao Google Drive
drive.mount('/content/drive')

# 2. Localizar o ZIP no Google Drive
ZIP = Path('/content/drive/MyDrive/metropt+3+dataset.zip')

if not ZIP.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ZIP}. Confira a pasta no Drive."
    )

# 3. Preparar a pasta de dados do projeto
pasta_dados = BASE / 'dados'
pasta_dados.mkdir(parents=True, exist_ok=True)

CSV = pasta_dados / 'MetroPT3(AirCompressor).csv'

# 4. Extrair o CSV do ZIP
with zipfile.ZipFile(ZIP, 'r') as arquivo:
    arquivos_csv = [
        nome for nome in arquivo.namelist()
        if nome.lower().endswith('.csv')
    ]

    print("Arquivos CSV encontrados:", arquivos_csv)

    if not arquivos_csv:
        raise ValueError("Nenhum CSV encontrado dentro do ZIP.")

    with arquivo.open(arquivos_csv[0]) as origem:
        with open(CSV, 'wb') as destino:
            shutil.copyfileobj(origem, destino)

# 5. Confirmar o carregamento
print("✅ Base de dados carregada!")
print("Caminho:", CSV)
print("Tamanho:", round(CSV.stat().st_size / 1e6, 2), "MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Arquivos CSV encontrados: ['MetroPT3(AirCompressor).csv']
✅ Base de dados carregada!
Caminho: /content/railpulse-metropt3/dados/MetroPT3(AirCompressor).csv
Tamanho: 218.3 MB


## 1. Conferir a proveniência, as lacunas e o conteúdo

In [7]:
import subprocess,sys
subprocess.run([sys.executable,'scripts/auditar_csv.py','--csv',str(CSV)],check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/auditar_csv.py', '--csv', '/content/railpulse-metropt3/dados/MetroPT3(AirCompressor).csv'], returncode=0)

## 2. Executar a análise completa dos ciclos (Etapa 4)

In [8]:
OUT=BASE/'resultados'/'etapa4'
OUT.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'analises/etapa4_ciclos.py','--csv',str(CSV),'--out',str(OUT)],check=True)

CompletedProcess(args=['/usr/bin/python3', 'analises/etapa4_ciclos.py', '--csv', '/content/railpulse-metropt3/dados/MetroPT3(AirCompressor).csv', '--out', '/content/railpulse-metropt3/resultados/etapa4'], returncode=0)

## 3. Conferir os valores com o artigo

In [9]:
import pandas as pd
from IPython.display import display
comparacao=pd.read_csv(OUT/'comparacao_regras.csv')
parametros=pd.read_json(OUT/'parametros.json',typ='series')
display(comparacao)
print('Tempo de recuperação P99:',parametros['q99_tempo_ate_9_7_s'])
print('Ciclos:',parametros['n_fases_carga'],'elegíveis:',parametros['n_ciclos_inicio_nominal_validos'])
assert parametros['original_n']==1516948
assert parametros['n_fases_carga']==15769
assert parametros['n_ciclos_inicio_nominal_validos']==10417
for metodo,n in [('Pressao_nao_recuperada_q99',33),('Carga_longa_q99',21),('Somente_carga_mesmo_limiar',136)]:
    assert int(comparacao.loc[comparacao.metodo==metodo,'total_alertas_teste'].iloc[0])==n
print('✅ Os 190 alertas das três regras foram reproduzidos.')

,metodo,total_alertas_teste,alertas_pre2h_falhas,alertas_durante_falhas,alertas_fora_proximidades,eventos_com_alerta_pre2h,alertas_fora_por_30_dias
0,Carga_longa_q99,21,7,1,13,3,3.86
1,Pressao_nao_recuperada_q99,33,13,0,20,3,5.94
2,Somente_carga_mesmo_limiar,136,18,0,115,3,34.16


Tempo de recuperação P99: 188.0
Ciclos: 15769 elegíveis: 10417
✅ Os 190 alertas das três regras foram reproduzidos.


## 4. Verificação de causalidade temporal

In [ ]:
subprocess.run([sys.executable,'analises/verificar_etapa4.py','--csv',str(CSV),'--out',str(OUT)],check=True)